<img src="https://data-analytics-fullstack-assets.s3.eu-west-3.amazonaws.com/M05-Exploratory_Data_Analysis/M3_D2_Streamforge.png" />

# StreamForge engagement report 📺

You work as a junior data analyst at **StreamForge**, a streaming platform that operates in seven countries.

The growth team needs a weekly engagement report. They want to know which titles users actually watch, how engagement changes by country and by genre, and where the platform has the most playback errors. The product team will use this report to decide which shows to promote next month.

The data lives in three CSV exports. Each export has its own gaps and quirks. Your job is to combine the files, deal with the missing values, and produce the answers the growth team needs.

## Files you will use

- `StreamForge_viewing_events.csv`: one row per playback event (1,000 rows). Each row records what a user did on a title at a given time.
- `StreamForge_user_profiles.csv`: one row per user (50 rows). Plan, age, country, and monthly revenue.
- `StreamForge_content.csv`: one row per title (100 rows). Genre, content type, and rating.

<Note type="important">

Download the three CSV files from the resources section before you start. Place them in the same folder as this notebook.

</Note>

## Setup

Run the cell below to import Pandas and NumPy. You will use these two libraries through the whole exercise.

In [2]:
# import libraries
import pandas as pd
import numpy as np

## Task 1: Load the three files

Before you can analyze anything, you need to read the data into memory. The growth team wants an honest look at the source files. They do not want you to clean anything yet. Loading the files first lets you see what you are working with.

Read each of the three CSV files into its own DataFrame. Name them `events`, `users`, and `content`. Then print the shape of each DataFrame and look at the first five rows. Confirm that the column names match what you expect from the file descriptions above.

You should end up with three separate DataFrames sitting in memory, ready for analysis.

In [4]:
# Load the data
content_df = pd.read_csv('files/StreamForge_content.csv')
users_df = pd.read_csv('files/StreamForge_user_profiles.csv')
events_df = pd.read_csv('files/StreamForge_viewing_events.csv')

# print the shapes of the dataframes
display(content_df.shape)
display(users_df.shape)
display(events_df.shape)

(100, 5)

(50, 5)

(1000, 11)

In [120]:
# Look at the first few rows of events
display(events_df.head())

,user_id,timestamp,event_type,duration_seconds,watched_duration,total_duration,content_id,episode_number,watched_seconds,total_seconds,user_rating
0,user_041,2024-02-05 07:14:00,play,1443,1186,1862,content_015,1,1186,1862.0,3.7
1,user_035,2024-02-24 01:01:00,error,251,80,1203,content_012,1,80,1203.0,3.0
2,user_002,2024-03-24 22:34:00,pause,919,316,2758,content_072,1,316,2758.0,NaN
3,user_001,2024-03-30 13:21:00,pause,629,141,1510,content_098,1,141,1510.0,NaN
4,user_022,2024-02-18 03:22:00,play,3117,2018,6422,content_014,1,2018,6422.0,1.2


In [121]:
# Look at the first few rows of users
display(users_df.head())

,user_id,subscription_type,user_age,country,monthly_revenue
0,user_001,premium,32.0,DE,13.99
1,user_002,standard,68.0,CA,8.99
2,user_003,premium,35.0,NaN,17.99
3,user_004,premium,56.0,JP,8.99
4,user_005,premium,21.0,DE,13.99


In [122]:
# Look at the first few rows of content
display(content_df.head())

,content_id,genre,primary_genre,content_type,rating
0,content_001,action,action,series,7.2
1,content_002,documentary,documentary,documentary,9.5
2,content_003,documentary,documentary,series,7.0
3,content_004,drama,drama,documentary,7.3
4,content_005,documentary,documentary,movie,7.1


## Task 2: Find the missing values

Real data is never complete. Some users skip the optional fields when they sign up. Some playback events lose data because of network problems. Before you compute anything, you need to know where the gaps are.

Use `isnull()` together with `sum()` to count the missing values in each column of each DataFrame. Print the result for the three DataFrames. Then write one short sentence in a Markdown cell that describes what you found. Mention which columns have gaps and roughly how many rows are affected.

<Note type="hint">

Calling `df.isnull()` returns a DataFrame of `True` and `False` values. Calling `.sum()` on that turns each column into a count of missing entries. So `df.isnull().sum()` gives you one number per column.

</Note>

In [123]:
# Missing values in events
missing_events = events_df.isnull().sum()
print(missing_events)

user_id               0
timestamp             0
event_type            0
duration_seconds      0
watched_duration      0
total_duration        0
content_id            0
episode_number        0
watched_seconds       0
total_seconds        50
user_rating         300
dtype: int64


In [124]:
# Missing values in users
missing_users = users_df.isnull().sum()
print(missing_users)

user_id              0
subscription_type    0
user_age             6
country              3
monthly_revenue      0
dtype: int64


In [125]:
# Missing values in content
missing_contents = content_df.isnull().sum()
print(missing_users)

user_id              0
subscription_type    0
user_age             6
country              3
monthly_revenue      0
dtype: int64


**What I found:**
...

## Task 3: Decide what to do with each gap

Now you know where the missing values are. The next step is to decide how to handle each one. There is no single right answer. The decision depends on what you want to measure.

The growth team will ask you to compute completion rates and average watch time later. Read the rules below and apply each one to the right column.

- First rule: when `total_seconds` is missing in `events`, you cannot compute completion for that row. Drop those rows. Use `dropna()` with the `subset` parameter. Save the result as `events_clean`.

- Second rule: when `user_rating` is missing in `events`, the user simply did not rate the title. That is normal user behavior, not a data error. Leave the `NaN` values in place. Pandas aggregation functions like `mean()` ignore `NaN` automatically.

- Third rule: when `user_age` is missing in `users`, fill the gap with the median age. Median is safer than mean because it does not move when a few users have extreme ages. Use `fillna()`. Save the result as `users_clean`.

- Fourth rule: when `country` is missing in `users`, you cannot guess where they live. Leave the `NaN` in place for now. You will handle this case when you produce country-level reports.

After you apply the rules, print the shape of `events_clean` and `users_clean`. Confirm that the missing values are gone from the columns you cleaned and still present in the columns you left alone.

<Note type="hint">

`dropna(subset=["col_name"])` removes only the rows where `col_name` is missing. Other missing values stay.

`fillna(value)` replaces every `NaN` in the chosen column with the value you give it.

</Note>

In [126]:
# Rule 1: drop events without total_seconds
events_clean_df = events_df.dropna(subset='total_seconds')
display(events_clean_df.shape)



(950, 11)

In [127]:
# Rule 3: fill missing user_age with the median age
median_age = users_df['user_age'].median()

# Create a copy of users to avoid modifying the original dataframe
users_clean_df = users_df.copy()
users_clean_df['user_age'] = users_clean_df['user_age'].fillna(median_age)




In [129]:
# Confirm: missing values gone where we cleaned, still present where we kept them
print("events_clean missing values:")
print(events_clean_df.isnull().sum())
print("users_clean missing values:")
print(users_clean_df.isnull().sum())

events_clean missing values:
user_id               0
timestamp             0
event_type            0
duration_seconds      0
watched_duration      0
total_duration        0
content_id            0
episode_number        0
watched_seconds       0
total_seconds         0
user_rating         290
dtype: int64
users_clean missing values:
user_id              0
subscription_type    0
user_age             0
country              3
monthly_revenue      0
dtype: int64


## Task 4: Summarize events by user, by content, and by event type

The growth team wants three quick numbers from the cleaned event data. Each one uses `groupby()` followed by `agg()`.

- First, find the top 10 users by total `watched_seconds`. Group `events_clean` by `user_id`, sum the `watched_seconds` column, and sort the result. The growth team uses this list to decide who receives the loyalty offer.

- Second, find the top 10 titles by total `watched_seconds`. For each title also report the number of sessions it had and the average `user_rating`. You need three aggregations on the same group, so use the `agg()` form that takes a dictionary or named keyword arguments. Sort by total watched time. The product team uses this list to plan promotions.

- Third, count how many events of each `event_type` happened on the platform. The engineering team uses this to track playback errors and buffer events.

Print all three results.

<Note type="hint">

When you need several aggregations at once, you can name each output column directly. For example:

```python
summary = events_clean.groupby("content_id").agg(
    total_watched=("watched_seconds", "sum"),
    session_count=("watched_seconds", "count"),
    avg_rating=("user_rating", "mean"),
)
```

This gives you one row per content with three named columns. The `mean` will silently skip any `NaN` in `user_rating`, which is what you want here.

</Note>

In [130]:
# Top 10 users by total watched seconds
watched_sec_per_userid = events_clean_df.groupby('user_id')['watched_seconds'].sum()
watched_sec_per_userid_sorted = watched_sec_per_userid.sort_values(ascending = False)
top10_users_sorted = watched_sec_per_userid_sorted.head(10)


#print(type(events_clean_df))

In [131]:
# Top 10 titles with three aggregations
"""contents_watched = events_clean_df.groupby('content_id').agg({
    'watched_seconds': 'sum',
    'content_id': 'count',
    'user_rating': 'mean'
})

# Sort by total_watched and print the top 10
contents_watched_sorted = contents_watched.sort_values('watched_seconds',ascending=False)
top10_contents_watched_sorted = contents_watched_sorted.head(10)
print(top10_contents_watched_sorted)"""

top10_contents_watched_sorted = (
    events_clean_df.groupby('content_id')
    .agg(
        total_watched=('watched_seconds', 'sum'),
        session_count=('content_id', 'count'),
        avg_rating=('user_rating', 'mean')
    )
    .sort_values(by='total_watched',ascending=False)
    .head(10)
)
print(top10_contents_watched_sorted)


             total_watched  session_count  avg_rating
content_id                                           
content_014          22978             15    2.690000
content_061          20167             23    3.092857
content_047          17998              9    2.383333
content_029          17672             11    2.557143
content_037          16937             11    2.866667
content_065          16568             15    3.500000
content_084          15726             10    2.650000
content_009          15608             17    2.578571
content_035          15334              8    4.740000
content_019          15178             13    3.007692


In [132]:
# Count of events per event_type
events_per_event_type = events_clean_df.groupby('event_type').size()
print(events_per_event_type)

event_type
buffer    186
error     189
pause     197
play      184
stop      194
dtype: int64


## Task 5: Enrich the events with user and content data

The summaries above only use the event file. To answer the growth team's real questions, you need to know which country each event came from and which genre each title belongs to. That information sits in the other two files.

Use `merge()` twice to bring the extra columns onto each event row.

- First, merge `events_clean` with `users_clean` on `user_id`. Use a left join so you keep every event even if a user record is missing. Save the result as `events_with_users`.

- Second, merge `events_with_users` with `content` on `content_id`. Again use a left join. Save the result as `events_full`.

Print the shape of `events_full` and the list of its columns. Confirm that you now have country, subscription type, primary_genre, and content_type sitting next to each event.

<Note type="hint">

A left join keeps all rows from the left DataFrame. Rows from the right DataFrame are matched on the join key. When there is no match, Pandas fills the new columns with `NaN`. The pattern is:

```python
merged = left_df.merge(right_df, on="key_column", how="left")
```

</Note>

In [133]:
# First merge: events + users
events_with_users = events_clean_df.merge(users_clean_df, on='user_id', how='left')
events_with_users.head()

,user_id,timestamp,event_type,duration_seconds,watched_duration,total_duration,content_id,episode_number,watched_seconds,total_seconds,user_rating,subscription_type,user_age,country,monthly_revenue
0,user_041,2024-02-05 07:14:00,play,1443,1186,1862,content_015,1,1186,1862.0,3.7,premium,26.0,FR,17.99
1,user_035,2024-02-24 01:01:00,error,251,80,1203,content_012,1,80,1203.0,3.0,standard,46.0,JP,8.99
2,user_002,2024-03-24 22:34:00,pause,919,316,2758,content_072,1,316,2758.0,NaN,standard,68.0,CA,8.99
3,user_001,2024-03-30 13:21:00,pause,629,141,1510,content_098,1,141,1510.0,NaN,premium,32.0,DE,13.99
4,user_022,2024-02-18 03:22:00,play,3117,2018,6422,content_014,1,2018,6422.0,1.2,standard,44.0,DE,13.99


In [134]:
# Second merge: add content metadata
events_full = events_with_users.merge(content_df, on='content_id', how='left')
events_full.columns

Index(['user_id', 'timestamp', 'event_type', 'duration_seconds',
       'watched_duration', 'total_duration', 'content_id', 'episode_number',
       'watched_seconds', 'total_seconds', 'user_rating', 'subscription_type',
       'user_age', 'country', 'monthly_revenue', 'genre', 'primary_genre',
       'content_type', 'rating'],
      dtype='str')

In [140]:
# Look at the first few rows of events_full
events_full.head()
events_full.shape

(950, 19)

## Task 6: Compare metrics across two dimensions

Now that every event carries the user and content attributes, you can answer questions that mix two dimensions. The growth team wants three multi-level views.

- First, total `watched_seconds` by country and by `primary_genre`. This shows which genres each country prefers. Sort the result and print the top 10 country and genre combinations.

- Second, average `user_rating` by `subscription_type` and by `content_type`. This shows whether premium users rate movies differently from basic users. Round the result to two decimals so it reads cleanly.
    
Third, count of events by `country` and by `event_type`. This shows where errors and buffer events concentrate. The engineering team needs this view to find the worst regions for streaming quality.

When you group by `country`, remember that some users have a missing country. Pandas will create a separate group for `NaN` rows. Decide whether you want to keep that group or drop it. Use `dropna(subset=["country"])` before the groupby if you decide to drop it.

<Note type="hint">

Pass a list of columns to `groupby()` to group by more than one dimension at a time:

```python
result = events_full.groupby(["country", "primary_genre"])["watched_seconds"].sum()
```

The output has a MultiIndex with one level per grouping column.

</Note>

In [143]:
# Watched seconds by country and primary_genre

# Drop rows with missing country before grouping

# Rule 1: drop events without total_seconds

# Group by country and primary_genre, sum watched_seconds

events_full_clean = events_full.dropna(subset=["country"])
result1 = (
    events_full_clean.groupby(["country", "primary_genre"])["watched_seconds"]
        .sum()
        .sort_values(ascending=False)
        .head(10)
)

display(result1)
events_full_clean.shape

country  primary_genre
BR       thriller         42926
         sci-fi           42238
JP       romance          34584
         drama            33278
         sci-fi           29837
UK       thriller         29636
DE       romance          27133
JP       comedy           24965
BR       drama            24083
JP       thriller         21964
Name: watched_seconds, dtype: int64

(893, 19)

In [160]:
# Average user_rating by subscription_type and content_type
result2 = (
    events_full.groupby(["subscription_type", "content_type"])["user_rating"]
        .mean()
        .round(2)
        #.sort_values(ascending=False)
)

display(result2)

# Round the result to two decimals


subscription_type  content_type
basic              documentary     2.90
                   movie           2.99
                   series          2.70
premium            documentary     3.00
                   movie           3.19
                   series          3.17
standard           documentary     2.92
                   movie           3.17
                   series          2.93
Name: user_rating, dtype: float64

In [166]:
# Event counts by country and event_type
result3 = (
    events_full.groupby(["country", "event_type"])["event_type"].count()
)

display(result3)

country  event_type
BR       buffer        35
         error         50
         pause         47
         play          40
         stop          39
CA       buffer        10
         error         16
         pause         18
         play          15
         stop          23
DE       buffer        24
         error         25
         pause         16
         play          21
         stop          18
FR       buffer        25
         error         15
         pause         25
         play          14
         stop          20
JP       buffer        45
         error         43
         pause         55
         play          42
         stop          45
UK       buffer        13
         error         10
         pause         16
         play          15
         stop          17
US       buffer        19
         error         20
         pause         11
         play          25
         stop          21
Name: event_type, dtype: int64